# بناء نظام RAG تقليدي باستخدام ChromaDB (الجزء 1 و 2)
### Building Traditional RAG with ChromaDB

دليل عملي مبسط يغطي:
1. تجهيز نصوص تجريبية وقراءتها بـ `DirectoryLoader`.
2. تقطيع النصوص لـ Chunks صغيرة بـ `RecursiveCharacterTextSplitter`.
3. توليد المتجهات (Embeddings).
4. تخزين الفهرس محلياً على القرص في `ChromaDB` (`persist_directory`).
5. البحث الدلالي البسيط والمقاس بدرجات المسافة ($L_2$ Distance).
6. إعادة استدعاء الفهرس المحفوظ مباشرة دون إعادة التضمين.

In [ ]:
import os
import shutil
from dotenv import load_dotenv

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

load_dotenv()
print("تم تحميل المكتبات بنجاح!")

## 1. تجهيز ملفات نصية تجريبية وحفظها محلياً

In [ ]:
docs = {
    "document_0.txt": (
        "Machine learning is a subset of artificial intelligence that learns from data. "
        "Common algorithms include linear regression, decision trees, and support vector machines."
    ),
    "document_1.txt": (
        "Deep learning uses multi-layer neural networks mimicking the human brain. "
        "CNNs excel in computer vision while RNNs and LSTMs process sequential and time-series data."
    ),
    "document_2.txt": (
        "Natural language processing enables machines to understand text. "
        "The Transformer architecture revolutionized NLP with self-attention mechanisms in models like BERT and GPT."
    )
}

sample_folder = "./chroma_sample_data"
os.makedirs(sample_folder, exist_ok=True)

for filename, content in docs.items():
    with open(os.path.join(sample_folder, filename), "w", encoding="utf-8") as f:
        f.write(content)

print(f"تم إنشاء {len(docs)} ملفات داخل: {sample_folder}")

## 2. قراءة الملفات دفعة واحدة باستخدام DirectoryLoader

In [ ]:
loader = DirectoryLoader(sample_folder, glob="*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
documents = loader.load()

print(f"عدد المستندات: {len(documents)}")
for doc in documents:
    name = os.path.basename(doc.metadata["source"])
    print(f"- {name}: {doc.page_content[:65]}...")

## 3. تقطيع النصوص إلى قطع صغيرة (Chunks)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=25)
chunks = splitter.split_documents(documents)

print(f"عدد الـ Chunks الناتجة: {len(chunks)}")
for i, chunk in enumerate(chunks, 1):
    print(f"  Chunk #{i}: {chunk.page_content}")

## 4. نموذج التضمين (Embeddings)

In [ ]:
openai_key = os.getenv("OPENAI_API_KEY", "").strip()

if openai_key and not openai_key.startswith("your_"):
    print("تضمينات OpenAI")
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
else:
    print("تضمينات Hugging Face السحابية")
    from langchain_huggingface import HuggingFaceEndpointEmbeddings
    hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN", os.getenv("HF_TOKEN", ""))
    embeddings = HuggingFaceEndpointEmbeddings(
        model="sentence-transformers/all-MiniLM-L6-v2",
        huggingfacehub_api_token=hf_token
    )

## 5. حفظ الـ Chunks في ChromaDB على القرص (Persistence)

In [ ]:
persist_dir = "./my_chroma_db"
if os.path.exists(persist_dir):
    shutil.rmtree(persist_dir, ignore_errors=True)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_dir,
    collection_name="rag_collection"
)
print(f"تم حفظ {len(chunks)} Chunks بنجاح في: {persist_dir}")

## 6. البحث الدلالي البسيط (Similarity Search)

In [ ]:
query = "Tell me about neural networks and deep learning"
results = vectorstore.similarity_search(query, k=2)

print(f"السؤال: {query}\n")
for i, doc in enumerate(results, 1):
    src = os.path.basename(doc.metadata["source"])
    print(f"نتيجة #{i} [{src}]:\n  {doc.page_content}\n")

## 7. البحث مع درجات المسافة (Similarity Search with Scores)
> ChromaDB تستخدم مسافة $L_2$ الإقليدية:
> - **الرقم الأقل = تشابه أكبر** (0 يعني تطابق كامل).

In [ ]:
query = "What is natural language processing and transformers?"
scored_results = vectorstore.similarity_search_with_score(query, k=3)

print(f"السؤال: {query}\n")
for i, (doc, score) in enumerate(scored_results, 1):
    src = os.path.basename(doc.metadata["source"])
    print(f"رتبة #{i} | المسافة (L2): {score:.4f} [{src}]:\n  {doc.page_content}\n")

## 8. استدعاء الفهرس المحفوظ من القرص مباشرة (بدون إعادة التضمين)

In [ ]:
reloaded_db = Chroma(
    persist_directory=persist_dir,
    embedding_function=embeddings,
    collection_name="rag_collection"
)

test_query = "decision trees and regression"
test_res = reloaded_db.similarity_search(test_query, k=1)
print(f"السؤال: {test_query}")
print(f"النتيجة من القرص: {test_res[0].page_content}")

## ملخص الخطوات:
- أنشأنا ملفات نصية وقرأناها بـ `DirectoryLoader`.
- جزأناها بـ `RecursiveCharacterTextSplitter`.
- حولناها لنواقل وخزناها في `ChromaDB` على القرص.
- بحثنا دلالياً واستعرضنا درجات المسافة $L_2$.
- أعدنا فتح الفهرس من القرص دون إعادة تضمين.

**المحاضرة القادمة (Part 3):** سنربط الفهرس مع LLM عبر LCEL لتوليد الإجابة النهائية.